In [1]:
pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [2]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
credit_approval = fetch_ucirepo(id=27) 
  
# data (as pandas dataframes) 
X = credit_approval.data.features 
y = credit_approval.data.targets 
  
# metadata 
print(credit_approval.metadata) 
  
# variable information 
print(credit_approval.variables) 


{'uci_id': 27, 'name': 'Credit Approval', 'repository_url': 'https://archive.ics.uci.edu/dataset/27/credit+approval', 'data_url': 'https://archive.ics.uci.edu/static/public/27/data.csv', 'abstract': 'This data concerns credit card applications; good mix of attributes', 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 690, 'num_features': 15, 'feature_types': ['Categorical', 'Integer', 'Real'], 'demographics': [], 'target_col': ['A16'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1987, 'last_updated': 'Wed Aug 23 2023', 'dataset_doi': '10.24432/C5FS30', 'creators': ['J. R. Quinlan'], 'intro_paper': None, 'additional_info': {'summary': 'This file concerns credit card applications.  All attribute names and values have been changed to meaningless symbols to protect confidentiality of the data.\r\n  \r\nThis dataset is interesting because there is a good mix of attributes --

In [4]:
import pandas as pd 
df = pd.concat([X, y], axis=1)

df.head()

,A15,A14,A13,A12,A11,A10,A9,A8,A7,A6,A5,A4,A3,A2,A1,A16
0,0,202.0,g,f,1,t,t,1.25,v,w,g,u,0.000,30.83,b,+
1,560,43.0,g,f,6,t,t,3.04,h,q,g,u,4.460,58.67,a,+
2,824,280.0,g,f,0,f,t,1.50,h,q,g,u,0.500,24.50,a,+
3,3,100.0,g,t,5,t,t,3.75,v,w,g,u,1.540,27.83,b,+
4,0,120.0,s,f,0,f,t,1.71,v,w,g,u,5.625,20.17,b,+


In [5]:
df.isnull().sum()

A15     0
A14    13
A13     0
A12     0
A11     0
A10     0
A9      0
A8      0
A7      9
A6      9
A5      6
A4      6
A3      0
A2     12
A1     12
A16     0
dtype: int64

In [7]:
# where missing value is numerical assigning median value
# Identify numerical columns
numerical_columns = df.select_dtypes(include=["number"]).columns

print(numerical_columns)
for column in numerical_columns:
    df[column] = df[column].fillna(df[column].median())

Index(['A15', 'A14', 'A11', 'A8', 'A3', 'A2'], dtype='object')


In [9]:
# handling categorical missing values
categorical_columns = df.select_dtypes(include=["object"]).columns
for column in categorical_columns:
    df[column] = df[column].fillna(df[column].mode()[0])

In [10]:
df.isnull().sum()

A15    0
A14    0
A13    0
A12    0
A11    0
A10    0
A9     0
A8     0
A7     0
A6     0
A5     0
A4     0
A3     0
A2     0
A1     0
A16    0
dtype: int64

### Preprocessing, Feature Selection and Engineering

In [11]:
## Separate the numeric and categorical columns
num_cols = df.select_dtypes(include='number').columns
cat_cols = df.select_dtypes(include='object').columns.drop('A16')

Outliers: remove rows where a numeric value is more than 3 standard deviations from the mean (Z-score method).

In [13]:
from scipy import stats
import numpy as np

# keep all numeric columns have |z-score| < 3
z_scores = np.abs(stats.zscore(df[num_cols]))
df = df[(z_scores < 3).all(axis=1)].reset_index(drop=True)

In [14]:
from sklearn.preprocessing import LabelEncoder

# one-hot encode categorical feature columns
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# label encode the binary target column
le = LabelEncoder()
df['A16'] = le.fit_transform(df['A16'])

## Categorical encoding: One-hot encode categorical features into separate 0/1 columns, while label encoding the binary target (A16).

In [15]:
## Scaling: scale numeric columns to a 0–1 range using MinMaxScaler, instead of standardizing them.
from sklearn.preprocessing import MinMaxScaler

# scale numeric features to a 0-1 range
scaler = MinMaxScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

In [16]:
df.head()

,A15,A14,A11,A8,A3,A2,A16,A13_p,A13_s,A12_t,...,A6_m,A6_q,A6_r,A6_w,A6_x,A5_gg,A5_p,A4_u,A4_y,A1_b
0,0.000000,0.297059,0.0625,0.108696,0.000000,0.332166,0,False,False,False,...,False,False,False,True,False,False,False,True,False,True
1,0.037066,0.063235,0.3750,0.264348,0.228718,0.873590,0,False,False,False,...,False,True,False,False,False,False,False,True,False,False
2,0.054541,0.411765,0.0000,0.130435,0.025641,0.209063,0,False,False,False,...,False,True,False,False,False,False,False,True,False,False
3,0.000199,0.147059,0.3125,0.326087,0.078974,0.273823,0,False,False,True,...,False,False,False,True,False,False,False,True,False,True
4,0.000000,0.176471,0.0000,0.148696,0.288462,0.124854,0,False,True,False,...,False,False,False,True,False,False,False,True,False,True


In [17]:
df.tail()

,A15,A14,A11,A8,A3,A2,A16,A13_p,A13_s,A12_t,...,A6_m,A6_q,A6_r,A6_w,A6_x,A5_gg,A5_p,A4_u,A4_y,A1_b
633,0.000000,0.382353,0.0000,0.108696,0.517179,0.142552,1,False,False,False,...,False,False,False,False,False,False,True,False,True,True
634,0.026079,0.294118,0.1250,0.173913,0.038462,0.173473,1,False,False,True,...,False,False,False,False,False,False,False,True,False,False
635,0.000066,0.294118,0.0625,0.173913,0.692308,0.223648,1,False,False,True,...,False,False,False,False,False,False,True,False,True,False
636,0.049643,0.411765,0.0000,0.003478,0.010513,0.081097,1,False,False,False,...,False,False,False,False,False,False,False,True,False,True
637,0.000000,0.000000,0.0000,0.720870,0.173077,0.413263,1,False,False,True,...,False,False,False,False,False,False,False,True,False,True


#### Model Creation and Evaluation


##### 4a. Classification Model
We split the data into train/test sets, train a Logistic Regression classifier, and evaluate it with two metrics: Accuracy and ROC-AUC.

In [18]:
from sklearn.model_selection import train_test_split

# separate features and target
X = df.drop(columns=['A16'])
y = df['A16']

# split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [19]:
## Training the model
from sklearn.linear_model import LogisticRegression

# train a Logistic Regression classifier
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

#### TEsting accuracy

In [20]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)

Accuracy: 0.8671875


In [22]:
from sklearn.metrics import roc_auc_score
auc = roc_auc_score(y_test, y_proba)
print("ROC-AUC:", auc)

ROC-AUC: 0.944944944944945


ROC-AUC: measures ranking quality regardless of the classification threshold, and is more robust to class imbalance

In [ ]:
ccuracy gives a simple overall correctness rate, but credit approval data is often imbalanced (more approvals or more rejections). ROC-AUC is threshold-independent and measures how well the model ranks positive vs. negative cases, making it more reliable when classes aren't perfectly balanced. Using both together gives a simple headline number (accuracy) alongside a more robust check (ROC-AUC).